# Tutorial: Compare NEON and EMIT data for SOAP site
## First of two notebooks
### Authors: Randi Neff, Hannah Rieder and Bridget Hass
#### last updated: 8/7/25

This tutorial is intended for Earth Science data professionals. Additional details and an overall summary of this project will be available on the [Earth Lab Blog](https://earthlab.colorado.edu/earth-data-analytics-professional-graduate-certificate/earth-data-analytics-certificate-cohorts). In this tutorial, we will learn how to compare resolution of two surface reflectance datasets by calculating Canopy Water Content (CWC) over individual tiles at the Soaproot Saddle (SOAP) field site in the Sierra National Forest in California. We will also evaluate forest health. One of the tiles we will compare was within the fire perimeter of the Creek Fire in fall 2020; the other tile will be adjacent to the burned one but was not burned by the Creek Fire. The hyperspectral data for the CWC calculation comes from the [National Ecological Observatory Network's (NEON) Level 3 Spectrometer orthorectified surface bidirectional reflectance - mosaic data product](https://data.neonscience.org/data-products/DP3.30006.002) and the [Earth Surface Mineral Dust Source Investigation (EMIT) L2A Reflectance Data Product](https://www.earthdata.nasa.gov/data/catalog/lpcloud-emitl2arfl-001).

## The objectives of this tutorial (divided between two notebooks) are to:
* Use co-located data from NEON and EMIT
* Calculate Canopy Water Content (CWC) from NEON and EMIT hyperspectral data
* Evaluate CWC data at different scales
* Compare between burned and unburned areas

DATA
The data provided with this tutorial were derived from existing code at:
* NEON Spectrometer orthorectified surface bidirectional reflectance data.
* Shapefiles for NEON burned and unburned tiles which are found in the DATA folder.
* EMIT L2A Estimated Surface Reflectance granule(s) that cover the NEON burned and unburned tiles.
* [Land Processes Distributed Active Archive Center (LP DAAC)](https://nasa.github.io/VITALS/python/03_EMIT_CWC_from_Reflectance.html#cwc-of-a-single-point).

Additional data will be downloaded programmatically within this tutorial.

## Tutorial Outline for Notebook 1 - NEON and EMIT Data 
* You will need a NEON user account, but will be provided with shapefiles for burned/unburned tile boundaries 
* NEON tile boundaries will be used to crop EMIT data to the same region of interest (ROI)
* You will need a NASA Earthdata account and functions found in the script folder

## Tutorial Outline for Notebook 2 - Canopy Water Content Comparison
* Open NEON and EMIT Reflectance Data
* Calculate Canopy Water Content (CWC)
* Compare CWC Datasets

## Notes on functions that are used in both notebooks and found in the scripts folder
* The two datasets (NEON & EMIT) are very large and in different formats so the aop_h5refl2xarray function does conversions to make them compatible
* The calc_ewt function was originally developed for EMIT data and some metadata is hardcoded so preserved while maintaining functionality
* The surfrfl_hvplot_image assists with visualizing both datasets

Additional considerations are discussed in the [README file](https://github.com/NEONScience/AOP-EMIT/blob/main/README.md)

## 0 Imports

In [ ]:
# Standard library imports
import csv # Working with tabular data
import math
import os, sys # Management of files and directories
import pathlib
# Some cells may generate warnings that we can ignore.
from zipfile import ZipFile # Handling data that comes in zipped formats
import warnings
warnings.filterwarnings('ignore')

# Third-party imports
import earthaccess # Accessing NASA Earth data
import folium # Visualizing geospatial data
import geopandas as gpd # Add geometry to panda dataframes
import h5py # Work with NEON reflectance data
import holoviews as hv # Interactive visualizations
import hvplot.xarray # Graphing
import matplotlib.pyplot as plt
import netCDF4 as nc # Read and write NetCDF files
import neonutilities as nu # Work with NEON reflectance data
import numpy as np
import pandas as pd # Data manipulation with dataframes and 1D arrays 
import rasterio as rio # Raster library
import requests # Downloading data from online sources
import xarray as xr # Working with multi-dimensional arrays

from branca.element import Figure # Structuring the HTML output
from IPython.display import display # Working in Jupyter notebooks
from rasterio.plot import show, show_hist # Generates and displays a histogram 
# of the raster data

## 1. Setup

In the setup section, we will:
1. create directories to store the data and functions for this project,
2. download and import the extra necessary scripts needed for this tutorial, and
3. download shapefiles to crop EMIT data to the NEON region of interest (ROI)

### 1.1 Create Data and Scripts (modules) Directories

The directories we will make are: an overarching data directory, a reflectance data directory, a CWC data directory, and a modules directory. The overarching data directory will contain the reflectance and CWC data directories and a CSV file containing lab measurements of the complex refractive index of liquid water. The reflectance data directory should contain the two EMIT cropped datasets (one for the burned tile and one for the unburned tile) and the two NEON reflectance datasets (one for the burned tile and one for the unburned tile) created in Tutorial Notebook 01. All of these datasets should be NetCDF files.

The CWC data directory will be where we store the results of this tutorial notebook: the CWC calculations of the burned and unburned tiles calculated using the cropped EMIT and NEON reflectance data. The modules directory must be in the same folder as where you have these tutorial notebooks stored for some of the imported functions to work. In this modules directory, we will manually download some Python scripts (.py files). The scripts contain various functions we'll use in this tutorial. The CWC calculation functions (calc_ewt and calc_ewt_neon) are expecting the data and the k_liquid_water_ice.csv file to be stored in a directory that is at the same file level as where you have this notebook stored. In the cell below, the `data_dir = r"../data"` code ensures that the data directory will be at the same file level as where this tutorial notebook is stored.

Here is a visual of how the directory and file structure will look once these directories are created and files are downloaded:

```
project-root/
│
├── data/                     # Main folder for data
│   ├── cwc/                  # Subfolder for canopy water content (CWC) data genearted in tutorial_notebook_02
│   │   ├── emit_burn_cwc.nc               
│   │   ├── emit_burn_cwc.tif               
│   │   ├── emit_unburn_cwc.nc
│   │   ├── emit_unburn_cwc.tif                           
│   │   ├── neon_burn_cwc.nc               
│   │   ├── neon_burn_cwc.tif               
│   │   ├── neon_unburn_cwc.nc               
│   │   └── neon_unburn_cwc.tif          
│   │
│   ├── refl/                 # Subfolder for reflectance data generated in tutorial_notebook_01
│   │   ├── EMIT_L2A_RFL_20230731T205320_2321214_004.nc
│   │   ├── EMIT_L2A_RFL_20230731_SOAP.nc
│   │   ├── EMIT_L2A_RFL_20230731_SOAP_burned.nc
│   │   ├── EMIT_L2A_RFL_20230731_SOAP_unburned.nc
│   │   ├── emit_soap_burned.nc
│   │   ├── emit_soap_unburned.nc
│   │   ├── NEON_D17_SOAP_DP3_298000_4100000_burned.h5
│   │   ├── NEON_D17_SOAP_DP3_298000_4101000_unburned.h5
│   │   ├── neon_burn_refl.nc
|   |   └── neon_unburn_refl.nc
|   └── k_liquid_water_ice.csv
│
└── notebooks/                # Subfolder for modules and tutorial notebooks
    ├── modules/              # Subfolder for Python scripts for processing and analysis
    │   ├── .ipynb_checkpoints
    |   ├── __pycache__
    |   ├── __init__
    |   ├── ewt_tools.py
    |   ├── ewt_calc2.py
    |   └── test_functions.py
    ├── tutorial_notebook_01.ipynb
    └── tutorial_notebook_02.ipynb
```

In [ ]:
# Define the file path for the data directory
data_dir = r"../data"

# Create the data_dir if it doesn't already exist
if not os.path.exists(data_dir):
    os.makedirs(data_dir)
    print(f'data directory made here: {data_dir}')
else:
    print(f'data directory already exists here: {data_dir}')

In [ ]:
# List of directories names
dir_list = ["refl", "shapefiles"]

# Define path where the directories will be created
root_path = data_dir

# Create the directories in the dir_list
for dir_name in dir_list:
    full_path = os.path.join(root_path, dir_name)
    if not os.path.exists(full_path):
        os.makedirs(full_path)
        print(f'directory made here: {full_path}')
    else:
        print(f'directory already exists here: {full_path}')

In [ ]:
# Define the file path for the modules directory
modules_dir = r"../notebooks/modules"

# Create the modules_dir if it doesn't already exist
if not os.path.exists(modules_dir):
    os.makedirs(modules_dir)
    print(f'modules directory made here: {modules_dir}')
else:
    print(f'modules directory already exists here: {modules_dir}')

### 1.2 Download and Import Necessary Scripts

The import code below expects the script to be in the modules directory we just created above.
* Manually download the three raw .py files from the [modules_dir](https://github.com/NEONScience/AOP-EMIT/tree/main/notebooks/modules) to your computer and move them to the `modules_dir` we created above.
* Run the code in the cell below to import functions into this notebook.

In [ ]:
# Import functions from the python scripts in the modules directory
from modules.emit_tools import emit_xarray # Open EMIT datasets into xarray.Dataset
from modules.test_functions import surfrfl_hvplot_image # Graphic display function

If not already installed, install the neonutilities packages 
using pip as follows:

`!pip install neonutilities`

`!pip install python-dotenv`

For this notebook:
* NEON & EMIT co-located data are provided
* EMIT L2A Reflectance granule is downloaded using earthaccess using an URL

In [ ]:
import neonutilities as nu
import dotenv

### Download the SOAP Burned and Unburned Reflectance Tiles:

In section **2.2 NEON Data** later in this tutorial notebook, we will read in NEON Reflectance .h5 files to then be processed. Click each of the links below to download NEON Reflectance .h5 data files for the burned and unburned tiles. Then, manually move them to the ../data/refl/ folder we made above.

* Burned: <a href="https://storage.googleapis.com/neon-aop-provisional-products/2024/FullSite/D17/2024_SOAP_8/L3/Spectrometer/Reflectance/NEON_D17_SOAP_DP3_298000_4100000_bidirectional_reflectance.h5" class="link--button link--arrow">NEON_D17_SOAP_DP3_298000_4100000_burned.h5</a>

* Unburned: <a href="https://storage.googleapis.com/neon-aop-provisional-products/2024/FullSite/D17/2024_SOAP_8/L3/Spectrometer/Reflectance/NEON_D17_SOAP_DP3_298000_4101000_bidirectional_reflectance.h5" class="link--button link--arrow">NEON_D17_SOAP_DP3_298000_4101000_unburned.h5</a>

Login to your NASA Earthdata account and 
create a .netrc file using the login function from the earthaccess library. 
If you do not have an Earthdata Account, you can create one here.

In [ ]:
earthaccess.login(persist=True)

In [ ]:
url = 'https://data.lpdaac.earthdatacloud.nasa.gov/lp-prod-protected/EMITL2ARFL.001/EMIT_L2A_RFL_001_20230731T205320_2321214_004/EMIT_L2A_RFL_001_20230731T205320_2321214_004.nc'

Get an HTTPS Session using your earthdata login, set a local path to save the file, and download the granule asset - This may take a while, the reflectance file is approximately 1.8 GB.

In [ ]:
# Get requests https Session using Earthdata Login Info
fs = earthaccess.get_requests_https_session()
# Retrieve granule asset ID from URL (to maintain existing naming convention)
granule_asset_id = url.split('/')[-1]
# Define Local Filepath
fp = f'../data/REFL/{granule_asset_id}'
# Download the Granule Asset if it doesn't exist
if not os.path.isfile(fp):
    with fs.get(url,stream=True) as src:
        with open(fp,'wb') as dst:
            for chunk in src.iter_content(chunk_size=64*1024*1024):
                dst.write(chunk)

# 2.0 Data in NEON Region of Interest 
* Create a NEON API token following [this tutorial](https://www.neonscience.org/resources/learning-hub/tutorials/neon-api-tokens-tutorial).
* Load a shapefile of the NEON site boundaries and SOAP reflectance data.


In [ ]:
# dotenv.set_key(dotenv_path=".env",
# key_to_set="NEON_TOKEN",
# value_to_set="your-token-here") - Use NEON token generated from your account

In [ ]:
# Function to download data stored on the internet in a public url to a local file
def download_url(url,data_dir):
    if not os.path.isdir(data_dir):
        os.makedirs(data_dir)
    filename = url.split('/')[-1]
    r = requests.get(url, allow_redirects=True)
    file_object = open(os.path.join(data_dir,filename),'wb')
    file_object.write(r.content)

In [ ]:
# Download and Unzip the NEON Flight Boundary Shapefile 
neon_boundary_url = "https://www.neonscience.org/sites/default/files/AOP_flightBoxes_0.zip"
# Use download_url function to save the file to a directory
os.makedirs('../data/shapefiles/', exist_ok=True)
download_url(neon_boundary_url,'../data/shapefiles/')
# Unzip the file
with ZipFile(f"../data/shapefiles/{neon_boundary_url.split('/')[-1]}", 'r') as zip_ref:
    zip_ref.extractall('../data/shapefiles/')

In [ ]:
# Save NEON shapefiles to use for cropping
aop_flightboxes = gpd.read_file("../data/shapefiles/AOP_flightBoxes/AOP_flightboxesAllSites.shp")
aop_flightboxes.head()

In [ ]:
# Isolate specific NEON location for cropping EMIT
site_id = 'SOAP'
aop_flightboxes[aop_flightboxes.siteID == site_id]

## 2.1 EMIT Data

EMIT L2A Reflectance Data are distributed in a non-orthocorrected spatially raw NetCDF4 (.nc) format consisting of the data and its associated metadata. To work with this data, we will use the emit_xarray function from the emit_tools.py module included in the repository.

In [ ]:
# Open the NetCDF file in read mode and create a Dataset object
ds_nc = nc.Dataset(fp)
ds_nc # Print information about the file

In [ ]:
# This returns a NetCDF4.Variable object, which acts like a NumPy array
ds_nc['location'] 

## 2.1.2 Crop EMIT Data to ROI

To make the rest of this code run quicker and to make the CWC calculation less intensive, crop the EMIT granule to the SOAP flight boxes now. Taken from [Hannah's 07 notebook](https://github.com/NEONScience/AOP-EMIT/blob/hrieder/notebooks/exploratory/hr/07_hr_cwc_emit.ipynb)

In [ ]:
# Open a shapefile of the ROI
aop_flightboxes = gpd.read_file("../data/shapefiles/AOP_flightboxes")
soap_polygon = aop_flightboxes[aop_flightboxes.siteID == 'SOAP']
shape = soap_polygon
shape

In [ ]:
# Define EMIT file path
emit_fp = ("../data"
           "/refl"
           "/EMIT_L2A_RFL_001_20230731T205320_2321214_004.nc")

In [ ]:
# Learn about emit_xarray function
help(emit_xarray)

In [ ]:
# Open emit_fp
emit_ds = emit_xarray(
    # Filepath
    emit_fp,
    # Orthorectify the dataset
    ortho=True
).load()

#Check dataset
emit_ds

In [ ]:
# Crop emit_ds to SOAP flightboxes
emit_crop_SOAP_ds = emit_ds.rio.clip(
    # Crop to SOAP polygon geometry
    shape.geometry.values,
    # Crop to SOAP polygon CRS
    shape.crs,
    # Include all pixels touched by polygon
    all_touched=True)

In [ ]:
# Export emit_crop_SOAP_ds and save to filepath we can use in calc_ewt fxn
emit_crop_SOAP_ds.to_netcdf("../data/refl"
           "/EMIT_L2A_RFL_20230731_SOAP.nc")

# Define filepath
emit_crop_SOAP_fp = ("../data/refl"
           "/EMIT_L2A_RFL_20230731_SOAP.nc")

# Check filepath
emit_crop_SOAP_fp

In [ ]:
# Open emit_crop_SOAP_fp to check the file path
emit_crop_SOAP_ds = xr.open_dataset(emit_crop_SOAP_fp, decode_coords="all")

# Check dataset
emit_crop_SOAP_ds

In [ ]:
# Check emit_crop_SOAP_ds.reflectance 
emit_crop_SOAP_ds.reflectance

In [ ]:
# Check NaN values and reflectance values in general
# to make sure they're NaN values and not an unrealistic value
# from cropping and exporting above.
emit_crop_SOAP_ds.reflectance.plot.hist()

In [ ]:
# View surface reflectance of cropped area for wavelength closest to 850
emit_crop_SOAP_ds.sel(
    wavelengths=850,
    # Use nearest valid index value
    method='nearest').reflectance.plot()

The code below expects the shapefiles to be in the ../data/shapefiles/ folder we created in Section 1 above. The shapefiles are for the tile boundaries of the burned and unburned tiles we will be focusing on for the rest of the tutorial.
* Manually download the 8 shapefile files from the [AOP-EMIT GitHub Repo data folder](https://github.com/NEONScience/AOP-EMIT/tree/main/data) to your computer and move them to the ../data/shapefiles/ folder we created above.
* Run the code in the cell below to read the shapefiles into the notebook.

In [ ]:
# Define filepath for burned tiles
burned_tile_shp_fp = ('../data/shapefiles'
                   '/NEON_D17_SOAP_DPQA_298000_4100000_boundary.shp')

# Write burned tile boundary filepath to geodataframe
burned_tile_gdf = gpd.read_file(burned_tile_shp_fp)

# Check geodataframe
burned_tile_gdf

In [ ]:
# Define filepath for burned tiles
unburned_tile_shp_fp = ('../data/shapefiles'
                   '/NEON_D17_SOAP_DPQA_298000_4101000_boundary.shp')

# Write burned tile boundary filepath to geodataframe
unburned_tile_gdf = gpd.read_file(unburned_tile_shp_fp)

# Check geodataframe
unburned_tile_gdf

In [ ]:
# Crop EMIT granule to burned tile of interest
EMIT_L2A_RFL_20230731_SOAP_burned_ds = emit_crop_SOAP_ds.rio.clip(
    # Crop to burned tile polygon geometry
    burned_tile_gdf.geometry.values,
    # Crop to burned tile polygon CRS
    burned_tile_gdf.crs,
    # Include all pixels touched by polygon
    all_touched=True)
# Check emit_burn_ds
EMIT_L2A_RFL_20230731_SOAP_burned_ds

In [ ]:
# Crop EMIT granule to unburned tile of interest
EMIT_L2A_RFL_20230731_SOAP_unburned_ds = emit_crop_SOAP_ds.rio.clip(
    # Crop to unburned tile polygon geometry
    unburned_tile_gdf.geometry.values,
    # Crop to unburned tile polygon CRS
    unburned_tile_gdf.crs,
    # Include all pixels touched by polygon
    all_touched=True)
# Check emit_unburn_ds
EMIT_L2A_RFL_20230731_SOAP_unburned_ds

In [ ]:
# View surface reflectance of burned area for wavelength closest to 850
EMIT_L2A_RFL_20230731_SOAP_burned_ds.sel(
    wavelengths=850,
    # Use nearest valid index value
    method='nearest').reflectance.plot()

In [ ]:
# View surface reflectance of unburned area for wavelength closest to 850
EMIT_L2A_RFL_20230731_SOAP_unburned_ds.sel(
    wavelengths=850,
    # Use nearest valid index value
    method='nearest').reflectance.plot()

In [ ]:
# Export EMIT cropped to burned NEON tile area and save for calc_ewt fxn in notebook 2
EMIT_L2A_RFL_20230731_SOAP_burned_ds.to_netcdf("../data/refl/"
           "/emit_soap_burned.nc")

# Define filepath
EMIT_L2A_RFL_20230731_SOAP_burned_fp = ("../data/refl/"
           "emit_soap_burned.nc")

# Check filepath
EMIT_L2A_RFL_20230731_SOAP_burned_fp

In [ ]:
# Export EMIT cropped to unburned NEON tile area and save for calc_ewt fxn
EMIT_L2A_RFL_20230731_SOAP_unburned_ds.to_netcdf("../data/refl/"
           "/emit_soap_unburned.nc")

# Define filepath
EMIT_L2A_RFL_20230731_SOAP_unburned_fp = ("../data/refl/"
           "emit_soap_unburned.nc")

# Check filepath
EMIT_L2A_RFL_20230731_SOAP_unburned_fp

## 2.2 NEON Data

In [ ]:
# Define a function to inspect the internal structure of an HDF5 file.

def inspect_h5_structure(file_path):
    """
    Opens an HDF5 file and prints its complete internal structure, including
    the names and shapes of all datasets.
    
    Args:
        file_path (str): The path to the HDF5 file to inspect.
    """
    # Check if the provided file path exists on the system.
    if not os.path.exists(file_path):
        print(f"Error: The file '{file_path}' does not exist.")
        return

    try:
        with h5py.File(file_path, 'r') as hdf5_file:
            print(f"--- Inspecting HDF5 file: {file_path} ---")

            # Define a nested function to recursively traverse the HDF5 file's structure.
            def recurse_h5_structure(group, indent=0):
                """Recursively prints the structure of a group."""
                for key in group.keys():
                    item = group[key]
                    if isinstance(item, h5py.Group):
                        print('  ' * indent + f"GROUP: {key}")
                        recurse_h5_structure(item, indent + 1)
                    elif isinstance(item, h5py.Dataset):
                        print('  ' * indent + f"DATASET: {key} (Shape: {item.shape})")
                        if 'Wavelength' in key or 'wavelength' in key:
                            print('  ' * (indent + 1) + "--> This dataset's"
                                  " name contains 'wavelength'. Check its shape.")

            # Start the recursive traversal from the root of the HDF5 file.
            recurse_h5_structure(hdf5_file)

        # Print a footer to indicate the end of the inspection.
        print("\n--- Inspection Complete ---")
        # Provide a specific hint for what to look for in NEON reflectance data files.
        print("Look for a DATASET named 'Wavelength' or similar with a shape of (1000,).")

    # Catch any exceptions that might occur during file access or processing.
    except Exception as e:
        print(f"An error occurred while inspecting the file: {e}")

# # --- Example Usage ---
# # The path to NEON .h5 file.
# h5_file_path = r"C:\Users\stem2\Documents\Capstone\AOP-EMIT\notebooks\exploratory\rn\FINAL\data\refl\NEON_D17_SOAP_DP3_298000_4100000_burned.h5"

# # Call the function with the example file path to execute the inspection.
# inspect_h5_structure(h5_file_path)

In [ ]:
# Define function that reads in a NEON AOP reflectance dataset and outputs an 
# xarray object - this function transposes and flips the data about the y axis 
# so y coords can be ascending, to match the EMIT data
def aop_h5refl2xarray(h5_filename):
    """
    Reads a NEON AOP reflectance dataset from an HDF5 file, processes it,
    and returns an xarray Dataset.

    The processing includes:
    - Transposing the data to (y, x, wavelengths) order.
    - Flipping the y-axis to ensure ascending y-coordinates.
    - Extracting reflectance data, wavelengths, FWHM, and metadata.
    - Identifying and marking bad band windows.
    - Calculating spatial coordinates (x_coords, y_coords).

    Args:
        h5_filename (str): The path to the NEON AOP HDF5 reflectance file.

    Returns:
        xr.Dataset: An xarray Dataset containing the processed reflectance data
                    and associated metadata.
    """
    # Open the HDF5 file in read mode using a 'with' statement for proper resource management.
    with h5py.File(h5_filename) as hdf5_file:
        print('Reading in ', h5_filename)

        # Extract the site name, which is typically the first key in the HDF5 file.
        sitename = list(hdf5_file.keys())[0]  
        # Access the 'Reflectance' group within the site's data.
        h5_refl_group = hdf5_file[sitename]['Reflectance']
        # Get the 'Reflectance_Data' dataset.
        refl_dataset = h5_refl_group['Reflectance_Data']
        # Load the reflectance data into a NumPy array and cast it to float32.
        refl_array = refl_dataset[()].astype('float32')

        # Transpose the array from (wavelengths, y, x) to (y, x, wavelengths)
        # to align with typical image/spatial data conventions (rows, columns, bands).
        
        # transpose so that we have y, x, wavelengths (similar to lat, lon, wavelengths)
        refl_arrayT = np.transpose(refl_array, (1, 0, 2)) 
        refl_arrayT = refl_array[::-1, :, :]  # Flip the first axis (y)

        # Get the shape of the processed reflectance array.
        refl_shape = refl_arrayT.shape
        # Extract wavelength values from the HDF5 metadata.
        wavelengths = h5_refl_group['Metadata']['Spectral_Data']['Wavelength'][:] #.astype('float32')
        # Extract Full Width at Half Maximum (FWHM) values from the HDF5 metadata.
        fwhm = h5_refl_group['Metadata']['Spectral_Data']['FWHM'][:] #.astype('float32')

        # create dictionary containing metadata information
        metadata = {}
        metadata['shape'] = refl_shape

        metadata['no_data_value'] = float(
            refl_dataset.attrs['Data_Ignore_Value'])
        metadata['scale_factor'] = float(refl_dataset.attrs['Scale_Factor'])

        # Extract the scale factor & Scale the reflectance data by the scale factor - this is memory intensive though!
        # Can do it after the fact
        # scale_factor = float(refl_dataset.attrs['Scale_Factor'])
        # refl_array = refl_array.astype(float) / scale_factor

        # Extract bad band windows
        metadata['bad_band_window1'] = (
            h5_refl_group.attrs['Band_Window_1_Nanometers'])
        metadata['bad_band_window2'] = (
            h5_refl_group.attrs['Band_Window_2_Nanometers'])

        # Initialize good_wavelengths array with 1s
        good_wavelengths = np.ones_like(wavelengths) #, dtype='float32')

        # Mark wavelengths within the bad band windows as 0
        for bad_window in [metadata['bad_band_window1'], metadata['bad_band_window2']]:
            bad_indices = np.where((wavelengths >= bad_window[0]) & (wavelengths <= bad_window[1]))[0]
            good_wavelengths[bad_indices] = 0
        good_wavelengths[-10:] = 0 # the last 10 indices also tend to be noisy
        
        # Extract metadata
        metadata['projection'] = h5_refl_group['Metadata']['Coordinate_System']['Proj4'][()].decode('utf-8')
        metadata['spatial_ref'] = h5_refl_group['Metadata']['Coordinate_System']['Coordinate_System_String'][()].decode('utf-8')
        metadata['EPSG'] = int(h5_refl_group['Metadata']
                               ['Coordinate_System']['EPSG Code'][()])

        # Parse the 'Map_Info' string to extract spatial extent information.
        # The string is split by commas to get individual components.
        map_info = str(
            h5_refl_group['Metadata']['Coordinate_System']['Map_Info'][()]).split(",")
        # extract the resolution & convert to floating decimal number
        pixel_width = float(map_info[5])
        
        pixel_height = float(map_info[6])
        # extract the upper left-hand corner coordinates from mapInfo and cast to float
        x_min = float(map_info[3]); x_min = int(x_min)
        y_max = float(map_info[4]); y_max = int(y_max)
        
        # calculate the xMax and yMin values from the dimensions
        # xMax = left edge + (# of columns * resolution)",
        x_max = x_min + (refl_shape[1]*pixel_width); x_max = int(x_max)
        # yMin = top edge - (# of rows * resolution)",
        y_min = y_max - (refl_shape[0]*pixel_height); y_min = int(y_min)

        # Calculate UTM coordinates
        x_coords = np.linspace(x_min, x_max, num=refl_shape[1]).astype(float)
        y_coordsT = np.linspace(y_min, y_max, num=refl_shape[0]).astype(float) 
        # y coords are ascending since we flipped the y in the beginning

        # Create an xarray DataArray for the reflectance data.
        # Define dimensions ('y', 'x', 'wavelengths') and coordinates.
        refl_xrT = xr.DataArray(refl_arrayT, dims=["y", "x", "wavelengths"], name="reflectance",
               coords={"y": ("y", y_coordsT), "x": ("x", x_coords),
                       "wavelengths": ("wavelengths", wavelengths), 
                       "fwhm": ("wavelengths", fwhm),
                       "good_wavelengths": ("wavelengths", good_wavelengths)})
        
        # Create the Transposed dataset
        dsT = xr.Dataset({"reflectance": refl_xrT})
              
        # Add metadata as attributes
        for key, value in metadata.items():
            if key not in ['shape', 'extent', 'ext_dict']:
                dsT.attrs[key] = value

        return dsT

In [ ]:
# Read in NEON burned reflectance and convert to xarray.Dataset
burned_reflectance_da = aop_h5refl2xarray(
    # File path to the NEON reflectance data for the burned tile
    r"../data/refl/NEON_D17_SOAP_DP3_298000_4100000_bidirectional_reflectance.h5"
)

# View burned_reflectance_da
burned_reflectance_da


In [ ]:
# Read in NEON unburned reflectance and convert to xarray.Dataset
unburned_reflectance_da = aop_h5refl2xarray(
    # File path to the NEON reflectance data for the unburned tile
    r"../data/refl/NEON_D17_SOAP_DP3_298000_4101000_bidirectional_reflectance.h5"
)

# View unburned_reflectance_da
unburned_reflectance_da

In [ ]:
# Define a function to update a NEON xarray Dataset.
def update_neon_xr(neon_refl_ds):
    """
    Cleans and prepares a NEON AOP reflectance xarray Dataset for analysis.

    This function performs the following key steps:
    1. Replaces a specific fill value (-9999) with NaN (Not a Number) to
       facilitate proper data handling and visualization.
    2. Scales the reflectance data using a scale factor provided in the
       dataset's metadata to convert integer values to physical reflectance.
    3. Sets data in "bad bands" (e.g., atmospheric water vapor absorption regions)
       to NaN, effectively removing noisy or unreliable spectral data.
    4. Writes the Coordinate Reference System (CRS) to the xarray object's
       attributes using the EPSG code found in the metadata.

    Args:
        neon_refl_ds (xarray.Dataset): An xarray Dataset containing NEON
                                       reflectance data and metadata.

    Returns:
        xarray.Dataset: The updated xarray Dataset with cleaned and scaled
                        reflectance data and a defined CRS.
    """

    # Set fill values equal to np.nan to improve visualization
    neon_refl_ds.reflectance.data[neon_refl_ds.reflectance.data == -9999] = np.nan
    
    # Scale by the reflectance scale factor
    neon_refl_ds['reflectance'].data = ((neon_refl_ds['reflectance'].data) /
                                        (neon_refl_ds.attrs['scale_factor']))
    
    # Set "bad bands" (water vapor absorption bands and noisy bands) to NaN
    neon_refl_ds['reflectance'].data[:,:,neon_refl_ds['good_wavelengths'].data==0.0] = np.nan

    # Write the Coordinate Reference System (CRS) to the dataset.
    neon_refl_ds.rio.write_crs(f"epsg:{neon_refl_ds.attrs['EPSG']}", inplace=True)

    # Return the modified xarray Dataset.
    return neon_refl_ds

In [ ]:
# Further pre-processing w/ burned_reflectance_da
burned_reflectance_da = update_neon_xr(burned_reflectance_da)

In [ ]:
# Further pre-processing w/ unburned_reflectance_da
unburned_reflectance_da = update_neon_xr(unburned_reflectance_da)

In [ ]:
# Export burned_reflectance_da and save to filepath we can use in notebook 2
burned_reflectance_da.to_netcdf("../data/refl/neon_burn_refl.nc")

# Define filepath
neon_burn_refl_fp = ("../data/refl/neon_burn_refl.nc")

# Check filepath
neon_burn_refl_fp

In [ ]:
# Export unburned_reflectance_da and save to filepath we can use in notebook 2
unburned_reflectance_da.to_netcdf("../data/refl/neon_unburn_refl.nc")

# Define filepath
neon_unburn_refl_fp = ("../data/refl/neon_unburn_refl.nc")

# Check filepath
neon_unburn_refl_fp

# 3.0 Visualize Data

In [ ]:
# Plot burned dataset to check it was loaded back in correctly
surfrfl_hvplot_image(
    EMIT_L2A_RFL_20230731_SOAP_burned_ds.sel(
        wavelengths=850,
        # Use nearest valid index value
        method='nearest'),
    plottitle='SOAP Burned Tile EMIT Surface Reflectance, 850.1 nm')

In [ ]:
# Plot unburned dataset to check it was loaded back in correctly
surfrfl_hvplot_image(
    EMIT_L2A_RFL_20230731_SOAP_unburned_ds.sel(
        wavelengths=850,
        # Use nearest valid index value
        method='nearest'),
    plottitle='SOAP Unburned Tile EMIT Surface Reflectance, 850.1 nm')

In [ ]:
# Function to adjust NEON contrast for RGB images
def gamma_adjust(rgb_ds, bright=0.2, white_background=False):
    """
    Applies gamma correction to an xarray Dataset containing RGB reflectance data.

    Gamma correction is a non-linear operation used to encode and decode
    luminance values in images. This function adjusts the image's brightness
    and contrast to make it more visually appealing, especially for images
    with a narrow range of values.

    Args:
        rgb_ds (xarray.Dataset): An xarray Dataset with 'reflectance' data,
                                 typically with dimensions (y, x, bands).
        bright (float): A target brightness value (between 0 and 1). This
                        value is used to calculate the gamma exponent.
        white_background (bool): If True, sets NaN values to 1, which will
                                 appear as a white background in plots.
                                 If False, NaNs are handled by the plotting library.

    Returns:
        xarray.Dataset: The updated xarray Dataset with gamma-adjusted
                        reflectance data.
    """
    # Extract the reflectance data array from the xarray Dataset.
    array = rgb_ds.reflectance.data
    # Dynamically adjust the gamma based on the image's overall brightness.
    gamma = math.log(bright)/math.log(np.nanmean(array)) 
    # Create exponent for gamma scaling - can be adjusted by changing 0.2 
    scaled = np.power(np.nan_to_num(array,nan=1),np.nan_to_num(gamma,nan=1)).clip(0,1) # Apply scaling and clip to 0-1 range
    # Conditionally handle NaN values for the background.
    if white_background == True:
        scaled = np.nan_to_num(scaled, nan = 1) # Set NANs to 1 so they appear white in plots
    # Update the reflectance data in the original xarray Dataset with the scaled data.
    rgb_ds.reflectance.data = scaled
    # Return the modified Dataset.
    return rgb_ds

# Plot the RGB image of the burned SOAP tile
neon_burn_rgb = burned_reflectance_da.sel(wavelengths=[650, 560, 470], method='nearest')
neon_burn_rgb = gamma_adjust(neon_burn_rgb,white_background=True)
neon_burn_rgb.hvplot.rgb(y='y',x='x',bands='wavelengths',
                         xlabel='UTM x',ylabel='UTM y',
                         title='NEON AOP Reflectance RGB - SOAP Burned Tile',
                         frame_width=480, frame_height=480)

In [ ]:
# Function to adjust NEON contrast for RGB images
def gamma_adjust(rgb_ds, bright=0.2, white_background=False):
    array = rgb_ds.reflectance.data
    gamma = math.log(bright)/math.log(np.nanmean(array)) # Create exponent for gamma scaling - can be adjusted by changing 0.2 
    scaled = np.power(np.nan_to_num(array,nan=1),np.nan_to_num(gamma,nan=1)).clip(0,1) # Apply scaling and clip to 0-1 range
    if white_background == True:
        scaled = np.nan_to_num(scaled, nan = 1) # Set NANs to 1 so they appear white in plots
    rgb_ds.reflectance.data = scaled
    return rgb_ds

# Plot the RGB image of the burned SOAP tile
neon_unburn_rgb = unburned_reflectance_da.sel(wavelengths=[650, 560, 470], method='nearest')
neon_unburn_rgb = gamma_adjust(neon_unburn_rgb,white_background=True)
neon_unburn_rgb.hvplot.rgb(y='y',x='x',bands='wavelengths',
                         xlabel='UTM x',ylabel='UTM y',
                         title='NEON AOP Reflectance RGB - SOAP Unburned Tile',
                         frame_width=480, frame_height=480)

# Continue to Notebook 2 to the Analysis of Co-located Data

## 4.0 Finding Data for Other Locations
To find other locations where NEON and EMIT AOP data overlap, please see this 
notebook: 
* [https://github.com/NEONScience/AOP-EMIT/blob/main/notebooks/exploratory/bh/01_bh_find_collocated_neon_emit_data.ipynb](https://github.com/NEONScience/AOP-EMIT/blob/main/notebooks/exploratory/bh/01_bh_find_collocated_neon_emit_data.ipynb)

See these links for a complete listing of data products:
1. NEON: [https://data.neonscience.org/data-products/explore](https://data.neonscience.org/data-products/explore)
2. EMIT: [https://github.com/nasa/EMIT-Data-Resources](https://github.com/nasa/EMIT-Data-Resources)

NEON spatial data & maps can be found here: [https://www.neonscience.org/data-samples/data/spatial-data-maps](https://www.neonscience.org/data-samples/data/spatial-data-maps)